# MT-SimNPO Analysis

Results from cloud training runs on RunPod A40 48GB.  
Models: Llama-3.1-8B-Instruct fine-tuned on TOFU (forget10 split).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.2)
FIGDIR = Path('figures')
FIGDIR.mkdir(exist_ok=True)

## 1. Data

In [ ]:
# TOFU benchmark results (forget10 split, seed=0 for baselines)
# forget_truth_ratio: ↑ better (closer to oracle_retrain ~0.64 is good)
# model_utility:      ↑ better

tofu = {
    'oracle_retrain':    {'ftr': 0.6406, 'mu': 0.6471},
    'GradDiff':          {'ftr': 0.0000, 'mu': 0.6713},
    'NPO':               {'ftr': 0.5083, 'mu': 0.6374},
    'SimNPO':            {'ftr': 0.5230, 'mu': 0.6365},
    'MTSimNPO (mw=0.5)': {'ftr': 0.5246, 'mu': 0.6399},
    'MTSimNPO (mw=1.0)': {'ftr': np.mean([0.5317, 0.5268, 0.5229]), 'mu': np.mean([0.6427, 0.6447])},
    'MTSimNPO (mw=2.0)': {'ftr': 0.5240, 'mu': 0.6412},
}

# MT-Eval results — Multi-Turn Recovery Rate (MTRR, ↓ better)
# Evaluated on mt_val.jsonl (3 trained attack types: priming_v2, self_correction_v2, persona_switch_v2)
mt_eval = {
    'MTSimNPO (mw=0.5)':  {'mtrr': 0.7075, 'seeds': [0]},
    'MTSimNPO (mw=1.0)':  {'mtrr': np.mean([0.69, 0.735, 0.6675]), 'seeds': [0.69, 0.735, 0.6675]},
    'MTSimNPO (mw=2.0)':  {'mtrr': 0.615,  'seeds': [0]},
}

print('TOFU Results:')
for k, v in tofu.items():
    ftr = f"{v['ftr']:.4f}" if v['ftr'] is not None else 'TBD'
    mu  = f"{v['mu']:.4f}"  if v['mu']  is not None else 'TBD'
    print(f"  {k:<25} ftr={ftr}  mu={mu}")

print('\nMT-Eval Results (MTRR, lower=better):')
for k, v in mt_eval.items():
    print(f"  {k:<25} mtrr={v['mtrr']:.4f}")

## 2. TOFU: Forget Quality vs Model Utility

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

colors = {
    'oracle_retrain':    '#2ca02c',
    'GradDiff':          '#d62728',
    'SimNPO':            '#ff7f0e',
    'MTSimNPO (mw=0.5)': '#9467bd',
    'MTSimNPO (mw=1.0)': '#1f77b4',
    'MTSimNPO (mw=2.0)': '#17becf',
}
markers = {
    'oracle_retrain':    '*',
    'GradDiff':          's',
    'SimNPO':            'D',
    'MTSimNPO (mw=0.5)': 'o',
    'MTSimNPO (mw=1.0)': 'o',
    'MTSimNPO (mw=2.0)': 'o',
}

for name, vals in tofu.items():
    if vals['ftr'] is None or vals['mu'] is None:
        continue
    ax.scatter(vals['mu'], vals['ftr'],
               color=colors.get(name, 'gray'),
               marker=markers.get(name, 'o'),
               s=150, zorder=5, label=name)
    ax.annotate(name, (vals['mu'], vals['ftr']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)

# Oracle reference lines
oracle_ftr = tofu['oracle_retrain']['ftr']
oracle_mu  = tofu['oracle_retrain']['mu']
ax.axhline(oracle_ftr, color='#2ca02c', ls='--', lw=1, alpha=0.5, label='Oracle FTR')
ax.axvline(oracle_mu,  color='#2ca02c', ls='--', lw=1, alpha=0.5, label='Oracle MU')

ax.set_xlabel('Model Utility (↑)')
ax.set_ylabel('Forget Truth Ratio (↑ toward oracle)')
ax.set_title('TOFU: Forget Quality vs Utility')
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig(FIGDIR / 'tofu_scatter.png', dpi=150)
plt.show()

## 3. TOFU: Forget Truth Ratio Bar Chart

In [ ]:
methods_bar = ['GradDiff', 'NPO', 'SimNPO', 'MTSimNPO (mw=0.5)', 'MTSimNPO (mw=1.0)', 'MTSimNPO (mw=2.0)']
bar_colors_map = {**colors, 'NPO': '#8c564b'}
ftr_vals = [tofu[m]['ftr'] for m in methods_bar]
bar_colors = [bar_colors_map.get(m, 'gray') for m in methods_bar]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(methods_bar, ftr_vals, color=bar_colors, edgecolor='white', width=0.6)

# MTSimNPO mw=1.0 seed error bar
mw1_idx = methods_bar.index('MTSimNPO (mw=1.0)')
mw1_seeds = [0.5315, 0.5266, 0.5229]
mw1_std = np.std(mw1_seeds)
ax.errorbar(mw1_idx, np.mean(mw1_seeds), yerr=mw1_std,
            fmt='none', color='black', capsize=5, capthick=2, lw=2)

oracle_ftr = tofu['oracle_retrain']['ftr']
ax.axhline(oracle_ftr, color='#2ca02c', ls='--', lw=2, label=f'Oracle retrain ({oracle_ftr:.3f})')
ax.set_ylabel('Forget Truth Ratio (↑ better)')
ax.set_title('TOFU Forget Quality — forget10 split')
ax.set_ylim(0, 0.75)
ax.legend()
for bar, val in zip(bars, ftr_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(FIGDIR / 'tofu_ftr_bar.png', dpi=150)
plt.show()

## 4. MT-Eval: MTRR Ablation (mt_weight)

In [ ]:
mw_vals   = [0.5, 1.0, 2.0]
mtrr_vals = [0.7075, np.mean([0.69, 0.735, 0.6675]), 0.615]
mtrr_stds = [0.0,    np.std([0.69, 0.735, 0.6675]),  0.0]

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(mw_vals, mtrr_vals, yerr=mtrr_stds,
            marker='o', ms=10, lw=2, color='#1f77b4',
            capsize=6, capthick=2, label='MT-SimNPO (mean ± std, 3 seeds)')

ax.set_xlabel('mt_weight (λ)')
ax.set_ylabel('MTRR (↓ better)')
ax.set_title('MTRR vs mt_weight — MT-Eval val split')
ax.set_xticks(mw_vals)
ax.set_ylim(0.5, 0.85)
ax.legend()

for x, y in zip(mw_vals, mtrr_vals):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points',
                xytext=(8, 4), fontsize=10)
plt.tight_layout()
plt.savefig(FIGDIR / 'mtrr_vs_mw.png', dpi=150)
plt.show()

## 5. MT-Eval: Seed Variance (mw=1.0)

In [ ]:
seeds    = [0, 1, 2]
mtrr_s   = [0.69, 0.735, 0.6675]

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(seeds, mtrr_s, color='#1f77b4', edgecolor='white', width=0.5)
ax.axhline(np.mean(mtrr_s), color='navy', ls='--', lw=1.5,
           label=f'Mean = {np.mean(mtrr_s):.3f}')
ax.set_xlabel('Seed')
ax.set_ylabel('MTRR (↓ better)')
ax.set_title('Seed Variance — MT-SimNPO mw=1.0')
ax.set_xticks(seeds)
ax.set_ylim(0.5, 0.85)
ax.legend()
for s, v in zip(seeds, mtrr_s):
    ax.text(s, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(FIGDIR / 'mtrr_seed_variance.png', dpi=150)
plt.show()

## 6. Summary Table

In [ ]:
print(f"{'Method':<28} {'FTR':>8} {'MU':>8} {'MTRR val':>10} {'Transfer MTRR':>15}")
print('-' * 74)
rows = [
    # name,             ftr,    mu,     mtrr_val, transfer_mtrr
    ('oracle_retrain',    0.6406, 0.6471, None,   0.850),
    ('GradDiff',          0.0000, 0.6713, None,   0.003),
    ('NPO',               0.5083, 0.6374, None,   0.540),
    ('SimNPO',            0.5230, 0.6365, None,   0.883),
    ('MTSimNPO mw=0.5',   0.5246, 0.6399, 0.7075, 0.7075),
    ('MTSimNPO mw=1.0',   np.mean([0.5317,0.5268,0.5229]), np.mean([0.6427,0.6447]), np.mean([0.69,0.735,0.6675]), 0.690),
    ('MTSimNPO mw=2.0',   0.5240, 0.6412, 0.615,  0.615),
]
for name, ftr, mu, mtrr, tmtrr in rows:
    ftr_s   = f'{ftr:.4f}'   if ftr   is not None else 'TBD'
    mu_s    = f'{mu:.4f}'    if mu    is not None else 'TBD'
    mtrr_s  = f'{mtrr:.4f}'  if mtrr  is not None else 'N/A'
    tmtrr_s = f'{tmtrr:.4f}' if tmtrr is not None else 'N/A'
    print(f"{name:<28} {ftr_s:>8} {mu_s:>8} {mtrr_s:>10} {tmtrr_s:>15}")
print()
print('FTR  = Forget Truth Ratio (↑ toward oracle 0.64)')
print('MU   = Model Utility (↑ better)')
print('MTRR val = Multi-Turn Recovery Rate on val split, trained attacks (↓ better)')
print('Transfer MTRR = MTRR on test split, transfer attacks (↓ better)')
print()
print('Key finding: NPO/SimNPO pass TOFU but remain 54-88% vulnerable to unseen transfer attacks.')
print('GradDiff near-zero MTRR is a false positive — model utility is destroyed (FTR=0).')

## 7. Vulnerability Demo Results

*(Run after NPO training completes — see `scripts/run_vulnerability_demo.sh`)*

This section will show MTRR for pre_unlearning, oracle_retrain, GradDiff, NPO, SimNPO on the held-out test split to demonstrate that single-turn unlearning methods remain vulnerable to multi-turn probing.

In [ ]:
# Vulnerability demo results (test split = transfer + stress attacks)
# overall_mtrr_transfer: cot_decomposition + triangulation (transfer generalization)
# overall_mtrr_stress:   crescendo (adversarial escalation)
vuln_results_path = Path('../results/vulnerability')

vuln_methods = ['pre_unlearning', 'oracle_retrain', 'GradDiff', 'NPO', 'SimNPO']
vuln_transfer = {}
vuln_stress   = {}
for m in vuln_methods:
    p = vuln_results_path / f'{m}_mt_eval.json'
    if p.exists():
        d = json.loads(p.read_text())
        vuln_transfer[m] = d.get('overall_mtrr_transfer')
        vuln_stress[m]   = d.get('overall_mtrr_stress')
    else:
        vuln_transfer[m] = None
        vuln_stress[m]   = None

print('Vulnerability Demo — MTRR on test split:')
print(f"{'Method':<22} {'Transfer MTRR':>14} {'Stress MTRR':>12}")
print('-' * 50)
for m in vuln_methods:
    t = f'{vuln_transfer[m]:.3f}' if vuln_transfer[m] is not None else 'pending'
    s = f'{vuln_stress[m]:.3f}'   if vuln_stress[m]   is not None else 'pending'
    print(f'{m:<22} {t:>14} {s:>12}')

# Bar chart for transfer MTRR
ready = {m: v for m, v in vuln_transfer.items() if v is not None}
if ready:
    fig, ax = plt.subplots(figsize=(8, 4))
    cmap = {'pre_unlearning': '#d62728', 'oracle_retrain': '#2ca02c',
            'GradDiff': '#aec7e8', 'NPO': '#8c564b', 'SimNPO': '#ff7f0e'}
    names = list(ready.keys())
    vals  = [ready[m] for m in names]
    ax.bar(names, vals, color=[cmap.get(m, '#1f77b4') for m in names],
           edgecolor='white', width=0.6)
    for i, (n, v) in enumerate(zip(names, vals)):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    ax.set_ylabel('MTRR — transfer attacks (↓ better)')
    ax.set_title('Multi-Turn Vulnerability Demo — Test Split')
    ax.set_ylim(0, 1.1)
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.savefig(FIGDIR / 'vulnerability_demo.png', dpi=150)
    plt.show()
else:
    print('\nRun scripts/run_vulnerability_demo.sh first, then re-run this cell.')

## 8. Metric Decomposition: NEM / SemSim / ROUGE vs mt_weight

How does each sub-metric change as mt_weight increases? Reveals whether MTRR improvement is driven by entity suppression (NEM), semantic divergence (SemSim), or surface form (ROUGE).

In [ ]:
# Metric decomposition from val MT-eval results.
# All attack types give identical per-metric values (priming_v2/self_correction_v2/persona_switch_v2)
# so we use the single reported value per variant.
# mw=1.0: only seed2 sub-metrics available from HF; seeds 0,1 pending run_analysis_gpu.sh.

metric_data = {
    0.5: {'mtrr': 0.7075, 'avg_nem': 0.5043, 'avg_sem': 0.7189, 'avg_rouge': 0.4041},
    1.0: {'mtrr': np.mean([0.69, 0.735, 0.6675]),
          'avg_nem':   0.5194,   # seed0 val
          'avg_sem':   0.6942,
          'avg_rouge': 0.3855},
    2.0: {'mtrr': 0.615,  'avg_nem': 0.4721, 'avg_sem': 0.6561, 'avg_rouge': 0.3481},
}

mw_x = sorted(metric_data.keys())
metrics_to_plot = [
    ('mtrr',      'MTRR',   '#1f77b4', '-o'),
    ('avg_nem',   'NEM',    '#d62728', '--s'),
    ('avg_sem',   'SemSim', '#ff7f0e', '--^'),
    ('avg_rouge', 'ROUGE-L','#2ca02c', '--D'),
]

fig, ax = plt.subplots(figsize=(7, 4))
for key, label, color, fmt in metrics_to_plot:
    vals = [metric_data[mw][key] for mw in mw_x]
    ax.plot(mw_x, vals, fmt, color=color, ms=8, lw=2, label=label)

ax.set_xlabel('mt_weight (λ)')
ax.set_ylabel('Score (↓ better for MTRR)')
ax.set_title('Metric Decomposition — MT-SimNPO val split')
ax.set_xticks(mw_x)
ax.set_ylim(0.3, 0.85)
ax.legend(loc='upper right', fontsize=9)
ax.axhline(0.5, color='gray', ls=':', lw=1, alpha=0.5)
plt.tight_layout()
plt.savefig(FIGDIR / 'metric_decomposition.png', dpi=150)
plt.show()

print("Interpretation:")
print("  SemSim drops most steeply (0.72→0.66): semantic content suppressed most by mt_weight.")
print("  NEM drops modestly (0.50→0.47): entity recall partially retained even at mw=2.0.")
print("  ROUGE tracks NEM: surface-form suppression follows entity suppression.")

## 9. FTR vs MTRR Tradeoff

The core claim: MT-SimNPO maintains forget quality (FTR) while reducing multi-turn leakage (MTRR). Pareto scatter across methods and mt_weight settings. Lower-right = better.

In [ ]:
# FTR vs MTRR tradeoff scatter.
# MTRR = val split trained attacks for MT-SimNPO; transfer MTRR (test) for baselines.
# Update with consistent split after run_analysis_gpu.sh.

tradeoff_methods = {
    # name:              (ftr,   mtrr_val,  mtrr_transfer, color,    marker, note)
    'SimNPO':            (0.523, None,      0.883,         '#ff7f0e','D',   'baseline'),
    'NPO':               (0.508, None,      0.540,         '#8c564b','s',   'baseline'),
    'MT-SimNPO mw=0.5':  (0.524, 0.7075,   None,          '#9467bd','o',   ''),
    'MT-SimNPO mw=1.0':  (np.mean([0.5315,0.5266,0.5229]),
                                  np.mean([0.69,0.735,0.6675]),
                                            None,          '#1f77b4','o',   ''),
    'MT-SimNPO mw=2.0':  (0.524, 0.615,    None,          '#17becf','o',   ''),
}

fig, ax = plt.subplots(figsize=(7, 5))
for name, (ftr, mtrr_val, mtrr_transfer, color, marker, note) in tradeoff_methods.items():
    mtrr = mtrr_val if mtrr_val is not None else mtrr_transfer
    if mtrr is None or ftr is None:
        continue
    ax.scatter(ftr, mtrr, color=color, marker=marker, s=160, zorder=5)
    label = name + (' (transfer)' if mtrr_val is None else '')
    ax.annotate(label, (ftr, mtrr), textcoords='offset points',
                xytext=(6, 4), fontsize=8, color=color)

oracle_ftr = tofu['oracle_retrain']['ftr']
ax.axvline(oracle_ftr, color='#2ca02c', ls='--', lw=1, alpha=0.5,
           label=f'Oracle FTR ({oracle_ftr:.3f})')
ax.set_xlabel('Forget Truth Ratio (↑ toward oracle 0.64)')
ax.set_ylabel('MTRR (↓ better)')
ax.set_title('FTR vs MTRR Tradeoff\n(val trained for MT-SimNPO; test transfer for baselines)')
ax.legend(fontsize=8)
ax.annotate('', xy=(0.545, 0.55), xytext=(0.51, 0.72),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.text(0.538, 0.60, 'better', fontsize=8, color='gray', rotation=-40)
plt.tight_layout()
plt.savefig(FIGDIR / 'ftr_vs_mtrr_tradeoff.png', dpi=150)
plt.show()

In [ ]:
from collections import defaultdict

def load_examples(jsonl_path):
    p = Path(jsonl_path)
    if not p.exists():
        return None
    return [json.loads(line) for line in p.read_text().splitlines() if line.strip()]

examples_mw1 = load_examples('../results/MTSimNPO_mw1.0_seed0/mt_test_examples.jsonl')

if examples_mw1:
    entity_stats = defaultdict(lambda: {'n': 0, 'leaked': 0, 'nem': [], 'sem': []})
    for ex in examples_mw1:
        s = entity_stats[ex['author_name']]
        s['n'] += 1
        s['leaked'] += int(ex['leaked'])
        s['nem'].append(ex['nem'])
        s['sem'].append(ex['sem'])

    entity_leak_rate = {
        name: {'leak_rate': s['leaked']/s['n'],
               'avg_nem': np.mean(s['nem']), 'avg_sem': np.mean(s['sem']), 'n': s['n']}
        for name, s in entity_stats.items()
    }
    sorted_ents = sorted(entity_leak_rate.items(), key=lambda x: x[1]['leak_rate'], reverse=True)
    top_k = 20

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    top_names = [x[0].split()[-1] for x in sorted_ents[:top_k]]
    top_rates = [x[1]['leak_rate'] for x in sorted_ents[:top_k]]
    ax.barh(range(top_k), top_rates[::-1], color='#1f77b4', alpha=0.8)
    ax.set_yticks(range(top_k))
    ax.set_yticklabels(top_names[::-1], fontsize=8)
    ax.set_xlabel('Leak Rate (↓ better)')
    ax.set_title(f'Top-{top_k} Most Leaked Entities\nMT-SimNPO mw=1.0 (test split)')
    ax.axvline(np.mean(top_rates), color='red', ls='--', lw=1,
               label=f'mean={np.mean(top_rates):.3f}')
    ax.legend(fontsize=8)

    all_rates = [v['leak_rate'] for v in entity_leak_rate.values()]
    axes[1].hist(all_rates, bins=20, color='#1f77b4', edgecolor='white', alpha=0.8)
    axes[1].axvline(np.mean(all_rates), color='red', ls='--', lw=1.5,
                    label=f'mean={np.mean(all_rates):.3f}')
    axes[1].set_xlabel('Per-Entity Leak Rate')
    axes[1].set_ylabel('# Entities')
    axes[1].set_title('Leak Rate Distribution Across Entities')
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(FIGDIR / 'per_entity_leakage.png', dpi=150)
    plt.show()

    print(f"Entities evaluated: {len(all_rates)}")
    print(f"Mean leak rate:     {np.mean(all_rates):.3f} ± {np.std(all_rates):.3f}")
    print(f"Fully suppressed (rate=0): {sum(1 for r in all_rates if r == 0)}")
    print(f"Fully leaked     (rate=1): {sum(1 for r in all_rates if r == 1)}")
else:
    print("Per-example results not yet available.")
    print("Run scripts/run_analysis_gpu.sh on RunPod, then re-run this cell.")

## 11. Per-Entity Leakage Analysis

Which forget-set authors are hardest to suppress? Loaded from per-example JSONL produced by `run_analysis_gpu.sh --examples_output`.  
**Requires** `results/MTSimNPO_mw1.0_seed0/mt_test_examples.jsonl`.

In [ ]:
# Transfer MTRR: baselines (results/vulnerability/) vs MT-SimNPO (results/MTSimNPO_*/mt_test.json)
results_base = Path('../results')

transfer_results = {}
for m in ['pre_unlearning', 'oracle_retrain', 'GradDiff', 'NPO', 'SimNPO']:
    p = results_base / 'vulnerability' / f'{m}_mt_eval.json'
    if p.exists():
        transfer_results[m] = json.loads(p.read_text()).get('overall_mtrr_transfer')

for label, run_id in [('MT-SimNPO mw=0.5', 'MTSimNPO_mw0.5_seed0'),
                       ('MT-SimNPO mw=1.0', 'MTSimNPO_mw1.0_seed0'),
                       ('MT-SimNPO mw=2.0', 'MTSimNPO_mw2.0_seed0')]:
    p = results_base / run_id / 'mt_test.json'
    if p.exists():
        transfer_results[label] = json.loads(p.read_text()).get('overall_mtrr_transfer')

pending = [k for k, v in transfer_results.items() if v is None]
if pending:
    print(f"Pending (run scripts/run_analysis_gpu.sh): {pending}")

ready = {k: v for k, v in transfer_results.items() if v is not None}
if ready:
    cmap_t = {'pre_unlearning':'#d62728','oracle_retrain':'#2ca02c','GradDiff':'#aec7e8',
               'NPO':'#8c564b','SimNPO':'#ff7f0e',
               'MT-SimNPO mw=0.5':'#9467bd','MT-SimNPO mw=1.0':'#1f77b4','MT-SimNPO mw=2.0':'#17becf'}
    names = list(ready.keys())
    vals  = [ready[n] for n in names]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(names, vals, color=[cmap_t.get(n, 'gray') for n in names], edgecolor='white', width=0.6)
    for i, (n, v) in enumerate(zip(names, vals)):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_ylabel('Transfer MTRR — test split (↓ better)')
    ax.set_title('Transfer Attack Vulnerability: Baselines vs MT-SimNPO')
    ax.set_ylim(0, 1.1)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig(FIGDIR / 'transfer_mtrr_comparison.png', dpi=150)
    plt.show()
else:
    print("No MT-SimNPO transfer results yet. Run scripts/run_analysis_gpu.sh on RunPod first.")

## 10. Transfer MTRR: MT-SimNPO vs Baselines (test split)

Does training on priming/self-correction/persona-switch generalize to unseen attack types (cot_decomposition, triangulation)?  
**Requires** `scripts/run_analysis_gpu.sh` → `results/MTSimNPO_*/mt_test.json`.